# Õppetund 12 - Vestluse ajaloo vähendamine agendi märkmelehe abil

See märkmik demonstreerib, kuidas pikemate vestluste konteksti hallata, kasutades Microsoft Agent Frameworki. Vestluste kasvades suureneb ka tokenite arv — lõpuks ületades mudeli konteksti akna suuruse. Selle lahendame **konteksti kokkuvõtte mustri** ja **agendi märkmelehega**, mis võimaldab säilitada püsivmälu.

## Mida õpid:
1. **Miks konteksti haldamine on oluline**: Tokenite piirmäärade ja konteksti akende mõistmine
2. **Konteksti-teadlikud agendid**: Agentide loomine, kes haldavad oma vestluse konteksti
3. **Konteksti kokkuvõtte muster**: Tööriistade kasutamine vestluse ajaloo kokkusurumiseks
4. **Agendi märkmeleht**: Püsiv mälu, mis säilitub ka konteksti vähendamisel

## Eeltingimused:
- Azure OpenAI seadistamine koos keskkonnamuutujatega
- Varasemate tundide agentide põhimõistetega tutvumine


## Seadistamine


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Miks konteksti haldamine on oluline

Igal LLM-il on piiratud **konteksti aken** — maksimaalne tokenite arv, mida ta suudab ühe päringuga töödelda. Mitme vooru pikkuse vestluse käigus:

- **Tokenite arv kasvab lineaarselt** iga kasutaja sõnumi ja assistendi vastusega.
- **Sõnumi tokenid domineerivad kulu** tõttu, et kogu ajalugu saadetakse iga vooru järel uuesti.
- Lõpuks vestlus **ületab konteksti akna** ja mudel kas kärbib seda või annab vea.

### Konteksti haldamise strateegiad

| Strateegia | Kuidas see toimib | Kompromiss |
|---|---|---|
| **Kärpimine** | Kõige vanemate sõnumite mahavõtmine | Varasema konteksti kaotus |
| **Kokkuvõtete tegemine** | Vanemate sõnumite kokkuvõtte koostamine | Mõned detailid kaovad, aga põhipunktid säilivad |
| **Märkmete / välise mälu kasutamine** | Oluliste faktide salvestamine vestlusest väljaspool | Nõuab tööriistade kasutamist, kuid säilib ka vähenduse korral |

Selles märkmikus kombineerime **kokkuvõtteid** ja **märkmikuvõtet** nii, et agent suudab säilitada järjepidevust isegi siis, kui vestluse ajalugu on kokkusurutud.


## Kontekstitundliku agendi loomine


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Pika vestluse simuleerimine

Vaatame läbi mitmevoorulise vestluse, et näha, kuidas kontekst kuhjub. Agent peaks hoidma olulisi detaile (eelistused, eelarve, reisi kuupäevad) mitme vooru vältel ja näitama järjepidevust.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Pane tähele, kuidas agendi mõtted varasematest vestlustest säilivad — ta teab Jaapanist, sushist, templidest, fotograafiast, 3000-dollarilisest eelarvest, üksikreisist ja aprillikuust. Lühikeses vestluses töötab see hästi, kuid vestluse kasvades muutub kogu ajaloo uuesti saatmine kalliks.

Jätkame vestlust veel mitme ringiga, et näha konteksti kogunemist:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Konteksti kokkuvõtte muster

Kuna vestlus kasvab, saame kasutada **kokkuvõtete tööriista**, et kogunenud konteksti kokku võtta lühikeses vormis. Agent kutsub seda tööriista, et salvestada põhieelistused, nii et isegi kui vanemad sõnumid eemaldatakse, säilitatakse olulised andmed.

See muster on aluseks keerukamatele ajaloo vähendamise meetoditele:
1. Agent tuvastab vestlusest võtmefaktid
2. Kutsub kokkuvõtte tööriista, et need salvestada
3. Vanemaid sõnumeid võib turvaliselt eemaldada, kuna kokkuvõte kajastab olulist

Allpool määratleme tööriista `summarize_preferences`, mida agent saab kutsuda, et salvestada lühike kokkuvõte õpitust.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Kokkuvõte

Selles õppetükis õppisite, kuidas hallata konteksti pikaaegsetes agendi vestlustes, kasutades Microsoft Agent Frameworki:

### Peamised mõisted
- **Kontekstikujud on piiratud** — iga üksik vestluse ajaloo token maksab raha ja arvestatakse limiiti.
- **Kokkuvõttevahendid** võimaldavad agendil kokku kogutud konteksti lühidalt kokku võtta, vähendades tokeni kasutust ja säilitades olulise info.
- **Agendi märkmelehed** pakuvad püsivat välismälu, mis säilib ka pärast vestluse vähendamist.

### Mida sa ehitasid
- **Kontekstitundlik agent**, mis säilitab järjepidevuse mitme vooru vestlustes
- **Kokkuvõttevahendi** (`summarize_preferences`), mis salvestab olulisemad kasutaja andmed kompaktse formaadi kujul
- **Mitmevooruline vestlus**, mis näitab konteksti säilitamist ja muutuste käsitlemist

### Reaalse elu rakendused
- **Klienditeeninduse robotid**: mäletavad eelistusi pikkade tugisessioonide jooksul
- **Isiklikud assistendid**: jälgivad käimasolevaid projekte ilma konteksti uuesti selgitamata
- **Hariduslikud juhendajad**: hoiavad õpilase edenemist paljude suhtluste jooksul

### Järgmised sammud
- Rakenda täielik märkmelehe tööriist failipõhise püsivusega
- Lisa automaatne ajaloo kärpimine pärast kokkuvõtte tegemist
- Koosta vektorandmebaasidega semantilise mälu otsinguks
- Loo agendid, kes võivad vestlusi päeva hiljem täieliku kontekstiga jätkata


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastutusest loobumine**:  
Seda dokumenti on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi püüame täpsust, palun arvestage, et automatiseeritud tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Kriitilise tähtsusega teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta ühegi arusaamatuse või valesti tõlgendamise eest, mis tuleneb selle tõlke kasutamisest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
